# NB10 — car detection vs. confidence_threshold demo

Small, standalone demo (not part of the NB09 sweep machinery): segment
**only "car"** on 2 full-tile parking-lot images, sweeping SAM3's
`confidence_threshold` (`[0.05, 0.1, 0.2, 0.3, 0.5]`) while holding
`prob_thd=0.1` and window params (`slide_crop=1024`, `slide_stride=768`)
fixed at NB09's known-good baseline. Each `confidence_threshold` value is a
full sliding-window rerun (it's baked into the SAM3 processor object, not a
free post-hoc threshold like `prob_thd`) — 2 tiles x 5 thresholds = 10 runs.

Tiles: `dop20_32_472_5525_1_he` (dealership lot + residential street
parking, forest, buildings) and `dop20_32_473_5525_1_he` (rail corridor,
two separate car lots, warehouses, no station building).


## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX \u2192 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated \u2192 {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned \u2192 {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Inference: car-only, confidence_threshold sweep

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, json, torch, torch.nn.functional as F
import numpy as np
from pathlib import Path
from PIL import Image

sys.stdout.reconfigure(line_buffering=True)

DEVICE   = "cuda"
OUT_DIR  = Path("/kaggle/working/output"); OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = OUT_DIR / "preds"; PRED_DIR.mkdir(parents=True, exist_ok=True)
BG_IDX = 255

# ── fixed params: only confidence_threshold varies. prob_thd and window size
# held at NB09's known-good baseline for full 5000x5000 tiles. ──
PROB_THD     = 0.1
SLIDE_CROP   = 1024
SLIDE_STRIDE = 768
CONF_THD_SWEEP = [0.05, 0.1, 0.2, 0.3, 0.5]

TILES = ["dop20_32_472_5525_1_he", "dop20_32_473_5525_1_he"]

def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    return hits[0] if hits else None

resolved_paths = {stem: find_tile(stem) for stem in TILES}
for stem, p in resolved_paths.items():
    print(f"Resolved {stem} -> {p}", flush=True)
if all(v is None for v in resolved_paths.values()):
    print("ERROR: no target tiles found under /kaggle/input.", flush=True)
    raise SystemExit(1)

from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

def make_processor(conf_thd):
    return Sam3Processor(model, confidence_threshold=conf_thd, device=DEVICE)

def cache_text(processor, words):
    cache = []
    with torch.no_grad():
        for word in words:
            te = model.backbone.forward_text([word], device=DEVICE)
            cache.append({k: v.cpu() for k, v in te.items()})
    return cache

def collect_class_scores(processor, state, h, w, te_cache, n_classes, device):
    logits = torch.zeros((n_classes, h, w), device=device)
    for cls_idx, te_cpu in enumerate(te_cache):
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state["backbone_out"][k] = v.to(device)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        scores = torch.zeros((h, w), device=device)
        if state.get("masks_logits") is not None and state["masks_logits"].shape[0] > 0:
            for i in range(state["masks_logits"].shape[0]):
                il = state["masks_logits"][i].squeeze()
                if il.shape != (h, w):
                    il = F.interpolate(il.view(1,1,*il.shape), size=(h,w),
                                       mode="bilinear", align_corners=False).squeeze()
                scores = torch.max(scores, il * state["object_score"][i])
        sem = state["semantic_mask_logits"].squeeze()
        if sem.shape != (h, w):
            sem = F.interpolate(sem.view(1,1,*sem.shape), size=(h,w),
                                mode="bilinear", align_corners=False).squeeze()
        scores = torch.max(scores, sem) * state["presence_score"]
        logits[cls_idx] = torch.max(logits[cls_idx], scores)
    return logits

def make_gaussian_kernel(h, w, dev):
    sy, sx = h/4.0, w/4.0
    y = torch.arange(h, device=dev).float() - (h-1)/2.0
    x = torch.arange(w, device=dev).float() - (w-1)/2.0
    return torch.exp(-y[:,None]**2/(2*sy**2)) * torch.exp(-x[None,:]**2/(2*sx**2))

def run_sliding_window(img_arr, words, processor, crop_size, stride):
    te_cache = cache_text(processor, words)
    n_cls = len(words)
    H_full, W_full = img_arr.shape[:2]

    h_grids = max(H_full - crop_size + stride - 1, 0) // stride + 1
    w_grids = max(W_full - crop_size + stride - 1, 0) // stride + 1
    total = h_grids * w_grids

    gauss_k = make_gaussian_kernel(crop_size, crop_size, DEVICE)
    acc     = torch.zeros(n_cls, H_full, W_full, device=DEVICE)
    wt_mat  = torch.zeros(H_full, W_full, device=DEVICE)

    for hi in range(h_grids):
        for wi in range(w_grids):
            y1 = hi*stride;  x1 = wi*stride
            y2 = min(y1+crop_size, H_full);  x2 = min(x1+crop_size, W_full)
            y1 = max(y2-crop_size, 0);       x1 = max(x2-crop_size, 0)

            crop_pil = Image.fromarray(img_arr[y1:y2, x1:x2])
            h_c, w_c = y2-y1, x2-x1

            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                state = processor.set_image(crop_pil)
                l = collect_class_scores(processor, state, h_c, w_c, te_cache, n_cls, DEVICE).float()

            g = gauss_k[:h_c, :w_c]
            acc[:, y1:y2, x1:x2] += l * g.unsqueeze(0)
            wt_mat[y1:y2, x1:x2] += g

            done = hi*w_grids + wi + 1
            print(f"    crop {done}/{total}", flush=True)

    return acc / wt_mat.unsqueeze(0)

def finalize(prob_map, prob_thd, bg_idx=BG_IDX):
    seg = prob_map.argmax(0)
    seg[prob_map.max(0)[0] < prob_thd] = bg_idx
    return seg.cpu().numpy()

manifest = []

def save_pred(stem, conf_thd, seg):
    tag = f"conf{conf_thd}"
    np.save(str(PRED_DIR / f"{stem}_{tag}.npy"), seg.astype(np.uint8))
    manifest.append(dict(stem=stem, tag=tag, prob_thd=PROB_THD, conf_thd=conf_thd,
                          slide_stride=SLIDE_STRIDE, slide_crop=SLIDE_CROP))
    print(f"  Saved pred: {tag}", flush=True)

for stem in TILES:
    img_path = resolved_paths[stem]
    if img_path is None:
        print(f"Skipping {stem} — file not found", flush=True)
        continue
    img_arr = np.array(Image.open(img_path).convert("RGB"))
    img_size = (img_arr.shape[1], img_arr.shape[0])
    print(f"\n=== {stem} ===", flush=True)

    for ct in CONF_THD_SWEEP:
        print(f"  [confidence_threshold={ct}] car-only, crop={SLIDE_CROP} stride={SLIDE_STRIDE}", flush=True)
        proc = make_processor(ct)
        logits = run_sliding_window(img_arr, ["car"], proc, SLIDE_CROP, SLIDE_STRIDE)
        seg = finalize(logits, PROB_THD)
        save_pred(stem, ct, seg)

    (PRED_DIR / f"{stem}_meta.json").write_text(json.dumps(dict(img_size=img_size)))

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\nWrote manifest.json with {len(manifest)} entries", flush=True)
PYEOF

## 4 — Metrics + rendering (GPU-free)

Reads the saved `.npy` label maps + `manifest.json` from the inference
cell above. Computes car pixel count and car blob count (connected
components) per `(tile, confidence_threshold)`, plots both against
`confidence_threshold`, and renders one RGB|overlay figure per config.


In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image
from scipy.ndimage import label
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR  = Path("/kaggle/working/output")
PRED_DIR = OUT_DIR / "preds"
BG_IDX = 255
CAR_IDX = 0  # single-class run: argmax over 1 class is always 0 where not background

def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    return hits[0] if hits else None

def to_rgb(seg, color=(255, 255, 0), bg_idx=BG_IDX):
    out = np.full((*seg.shape, 3), 30, dtype=np.uint8)
    out[seg == CAR_IDX] = color
    return out

def render_result(stem, img_arr, seg, out_path, conf_thd, alpha=0.6):
    fig = plt.figure(figsize=(20, 12))
    ax = fig.subplots(1, 2)
    ax[0].imshow(img_arr); ax[0].axis('off')
    ax[0].set_title(f"{stem}.jpg", fontsize=12, fontweight='bold')
    ax[1].imshow(img_arr)
    ax[1].imshow(to_rgb(seg), alpha=alpha)
    ax[1].axis('off')
    ax[1].set_title(f'car only, confidence_threshold={conf_thd} (α={alpha})',
                     fontsize=12, fontweight='bold')
    fig.text(0.5, 0.02, f"prob_thd=0.1    confidence_threshold={conf_thd}",
             ha='center', fontsize=10, family='monospace')
    fig.savefig(str(out_path), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  Rendered: {out_path.name}", flush=True)

manifest = json.loads((OUT_DIR / "manifest.json").read_text())
img_cache = {}
results = {}  # stem -> list of (conf_thd, pixel_count, blob_count)

for entry in manifest:
    stem, tag, conf_thd = entry["stem"], entry["tag"], entry["conf_thd"]
    if stem not in img_cache:
        img_cache[stem] = np.array(Image.open(find_tile(stem)).convert("RGB"))
    img_arr = img_cache[stem]
    seg = np.load(str(PRED_DIR / f"{stem}_{tag}.npy"))

    car_mask = (seg == CAR_IDX)
    pixel_count = int(car_mask.sum())
    _, blob_count = label(car_mask)
    results.setdefault(stem, []).append((conf_thd, pixel_count, blob_count))

    render_result(stem, img_arr, seg, OUT_DIR / f"{stem}_{tag}.png", conf_thd)

# punchline plot: pixel count + blob count vs confidence_threshold, per tile
for stem, rows in results.items():
    rows.sort()
    cts, pix, blobs = zip(*rows)
    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.plot(cts, pix, "o-", color="tab:blue", label="car pixel count")
    ax1.set_xlabel("confidence_threshold")
    ax1.set_ylabel("car pixel count", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(cts, blobs, "s--", color="tab:red", label="car blob count")
    ax2.set_ylabel("car blob count", color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    plt.title(f"{stem}: car detection vs confidence_threshold")
    fig.tight_layout()
    fig.savefig(str(OUT_DIR / f"{stem}_confthd_trend.png"), dpi=150)
    plt.close(fig)
    print(f"{stem}: {rows}")

print(f"\nRendered {len(manifest)} images + {len(results)} trend plots.", flush=True)